In [ ]:
# One time Only

!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils pyzbar opencv-python-headless
!apt-get install -y libzbar0

import os
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

print("=" * 60)
print("LEGAL METROLOGY (2011) - MODEL INITIALIZATION")
print("=" * 60)

if not torch.cuda.is_available():
    raise SystemError("CUDA GPU is not detected. Please select a T4 GPU in Runtime settings.")

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# Google Drive location
MODEL_PATH = "/content/drive/MyDrive/AI_Models/Qwen2.5-VL-7B-Instruct"

os.makedirs(MODEL_PATH, exist_ok=True)

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 768 * 28 * 28

print("\n[System] Loading Qwen2.5-VL-7B with 4-bit NF4 quantization...")
print(f"[Storage] {MODEL_PATH}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Download model from Hugging Face directly into Google Drive
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
    cache_dir=MODEL_PATH
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    cache_dir=MODEL_PATH
)

print("\n" + "=" * 60)
print("MODEL READY")
print("=" * 60)
print(f"Model cache stored in:")
print(MODEL_PATH)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 15.4 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'libzbar0t64' instead of 'libzbar0'
The following additional packages will be installed:
  fonts-droid-fallback fonts-noto-mono fonts-urw-base35 ghostscript
  imagemagick-6-common libdjvulibre-text libdjvulibre21 libgs-common libgs10
  libgs10-common libidn12 libijs-0.35 libimath-3-1-29t64 libjbig2dec0
  libjxr-tools libjxr0t64 liblqr-1-0 libmagickcore-6.q16-7-extra
  libmagickcore-6.q16-7t64 libmagickwand-6.q16-7t64 libopenexr-3-1-30
  libraw23t64 libv4l-0t64 libv4lconvert0t64 libwmflite-0.2-7 poppler-data
  xfonts-encodings xfonts-utils
Suggested packages:
  fo

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils

import os
import torch

from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

MODEL_PATH = "/content/drive/MyDrive/AI_Models/Qwen2.5-VL-7B-Instruct"

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 768 * 28 * 28

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading model from Hugging Face cache into T4...")

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    cache_dir=MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    cache_dir=MODEL_PATH,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS
)

print("✅ Qwen2.5-VL-7B loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model from Hugging Face cache into T4...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

✅ Qwen2.5-VL-7B loaded successfully!


In [ ]:
import os
import cv2
import torch
import base64
import nest_asyncio
import threading
import uvicorn

from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pyzbar.pyzbar import decode
from PIL import Image
from qwen_vl_utils import process_vision_info

nest_asyncio.apply()

app = FastAPI(
    title="Legal Metrology Compliance Scanner",
    description="AI-powered product label compliance scanner",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

OCR_PROMPT = """Read the product packaging label and extract the following mandatory declarations.
Format your response EXACTLY as shown below. If a detail is missing, unprinted, or blank, write "MISSING".

MANUFACTURER: [Name and complete address of the manufacturer, packer, or importer]
COMMODITY_NAME: [Common or generic name of the commodity]
NET_QUANTITY: [Net quantity in weight, volume, or number]
DATE: [Month and year of manufacture, packing, or import]
MRP: [Maximum Retail Price inclusive of all taxes]
CONSUMER_CARE: [Consumer care contact name, address, telephone number, and email]"""


def scan_barcode_bytes(image_bytes):
    image_array = cv2.imdecode(
        __import__("numpy").frombuffer(image_bytes, dtype=__import__("numpy").uint8),
        cv2.IMREAD_COLOR
    )

    if image_array is None:
        return []

    gray = cv2.cvtColor(image_array, cv2.COLOR_BGR2GRAY)

    variants = [
        ("original", image_array),
        ("grayscale", gray),
        (
            "otsu",
            cv2.threshold(
                gray,
                0,
                255,
                cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )[1]
        )
    ]

    detected = []

    for name, variant in variants:
        try:
            codes = decode(variant)

            for c in codes:
                data = c.data.decode(
                    "utf-8",
                    errors="ignore"
                )

                item = {
                    "type": c.type,
                    "data": data
                }

                if item not in detected:
                    detected.append(item)

        except Exception:
            continue

    return detected


def extract_compliance(image_bytes):

    image = Image.open(
        __import__("io").BytesIO(image_bytes)
    ).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": OCR_PROMPT
                }
            ]
        }
    ]

    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    )

    inputs = inputs.to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(
            inputs.input_ids,
            generated_ids
        )
    ]

    raw_output = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True
    )[0].strip()

    extracted_data = {
        "manufacturer_details": None,
        "commodity_name": None,
        "net_quantity": None,
        "date_of_manufacture": None,
        "retail_sale_price": None,
        "consumer_care": None
    }

    for line in raw_output.splitlines():

        line = line.strip()
        upper_line = line.upper()

        if upper_line.startswith("MANUFACTURER:"):
            extracted_data["manufacturer_details"] = \
                line.split(":", 1)[1].strip()

        elif upper_line.startswith("COMMODITY_NAME:"):
            extracted_data["commodity_name"] = \
                line.split(":", 1)[1].strip()

        elif upper_line.startswith("NET_QUANTITY:"):
            extracted_data["net_quantity"] = \
                line.split(":", 1)[1].strip()

        elif upper_line.startswith("DATE:"):
            extracted_data["date_of_manufacture"] = \
                line.split(":", 1)[1].strip()

        elif upper_line.startswith("MRP:"):
            extracted_data["retail_sale_price"] = \
                line.split(":", 1)[1].strip()

        elif upper_line.startswith("CONSUMER_CARE:"):
            extracted_data["consumer_care"] = \
                line.split(":", 1)[1].strip()

    rules_check = {
        "commodity_name":
            "Rule 6(1)(b): Generic/Common Commodity Name",

        "net_quantity":
            "Rule 6(1)(c): Net Quantity (Weight/Measure/Count)",

        "date_of_manufacture":
            "Rule 6(1)(d): Month & Year of Packing/Mfg",

        "retail_sale_price":
            "Rule 6(1)(e): Maximum Retail Price (MRP)",

        "manufacturer_details":
            "Rule 6(1)(a): Name & Address of Mfg/Packer",

        "consumer_care":
            "Rule 6(2): Consumer Complaint Contact Details"
    }

    missing_tokens = [
        "missing",
        "not found",
        "none",
        "null",
        "n/a",
        "",
        "[missing]"
    ]

    passed = []
    violations = []

    for key, rule_label in rules_check.items():

        value = extracted_data.get(key)

        if (
            value
            and str(value).strip().lower()
            not in missing_tokens
        ):
            passed.append({
                "field": key,
                "rule": rule_label,
                "value": value
            })

        else:
            violations.append({
                "field": key,
                "rule": rule_label,
                "value": None
            })

    compliant = len(violations) == 0

    return {
        "raw_output": raw_output,
        "extracted_data": extracted_data,
        "passed": passed,
        "violations": violations,
        "passed_count": len(passed),
        "violation_count": len(violations),
        "total_checks": 6,
        "compliant": compliant,
        "overall_evaluation":
            "COMPLIANT: All mandatory declarations are present."
            if compliant
            else
            "NON-COMPLIANT: Missing or incomplete statutory declarations."
    }


@app.get("/")
def root():
    return {
        "status": "online",
        "service": "Legal Metrology Compliance Scanner"
    }


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "cuda": torch.cuda.is_available(),
        "model_loaded": model is not None
    }


@app.post("/scan")
async def scan_product(
    file: UploadFile = File(...)
):

    if not file.content_type:
        raise HTTPException(
            status_code=400,
            detail="Invalid file."
        )

    if not file.content_type.startswith("image/"):
        raise HTTPException(
            status_code=400,
            detail="Please upload an image file."
        )

    image_bytes = await file.read()

    if not image_bytes:
        raise HTTPException(
            status_code=400,
            detail="Empty image."
        )

    print(
        f"\nScanning: {file.filename} "
        f"({len(image_bytes) / 1024:.1f} KB)"
    )

    barcodes = scan_barcode_bytes(image_bytes)

    print("Running Qwen2.5-VL...")

    compliance = extract_compliance(image_bytes)

    return {
        "filename": file.filename,
        "barcodes": barcodes,
        "barcode_detected": len(barcodes) > 0,
        "compliance": compliance
    }


print("FastAPI application created successfully.")

FastAPI application created successfully.


In [ ]:
import threading
import uvicorn

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health"
)

print(response.json())

INFO:     127.0.0.1:60770 - "GET /health HTTP/1.1" 200 OK
{'status': 'healthy', 'cuda': True, 'model_loaded': True}


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!cloudflared --version

In [ ]:
import subprocess
import re
import time

cloudflare_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://127.0.0.1:8000"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

PUBLIC_URL = None

for _ in range(60):
    line = cloudflare_process.stdout.readline()

    if line:
        print(line, end="")

        match = re.search(
            r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
            line
        )

        if match:
            PUBLIC_URL = match.group(0)
            break

    time.sleep(1)

if PUBLIC_URL:
    print("\n" + "=" * 70)
    print("✅ PUBLIC API READY")
    print("=" * 70)
    print(PUBLIC_URL)
    print()
    print("Health:", PUBLIC_URL + "/health")
    print("Scan:", PUBLIC_URL + "/scan")
else:
    print("❌ Tunnel URL was not detected.")

2026-09-10T14:47:35Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-10T14:47:35Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-10T14:47:42Z INF +--------------------------------------------------------------------------------------------+
2026-09-10T14:47:42Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-10T14:47:42Z INF |  https://wiley-heard-nevada-module.trycloudflare.com  

In [ ]:
import requests

response = requests.get(PUBLIC_URL + "/health", timeout=30)

print("Status:", response.status_code)
print(response.json())

INFO:     34.12.11.202:0 - "GET /health HTTP/1.1" 200 OK
Status: 200
{'status': 'healthy', 'cuda': True, 'model_loaded': True}
